# Mitsui Local: Train, Predict Holdout, Score

No Kaggle gateway. Trains on `date_id < HOLDOUT_START`, predicts the rest
day-by-day with a rolling buffer, scores against `train_labels`.

**Edit `DATA_DIR` below to point at wherever you unzipped the competition
data.**

In [2]:
import os
import numpy as np
import pandas as pd
from collections import deque
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Edit this to your local path
DATA_DIR = '/Users/hayden/coderepos_mac_mini/mitsui_commodity/data'
HOLDOUT_START = 1827

train        = pd.read_csv(f'{DATA_DIR}/train.csv')
train_labels = pd.read_csv(f'{DATA_DIR}/train_labels.csv')
target_pairs = pd.read_csv(f'{DATA_DIR}/target_pairs.csv')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"device: {device}")
print(f"train: {train.shape}, train_labels: {train_labels.shape}")


device: cpu
train: (1961, 558), train_labels: (1961, 425)


In [3]:
# Helpers

def pearson_loss(pred, target):
    pred_c   = pred   - pred.mean(dim=-1, keepdim=True)
    target_c = target - target.mean(dim=-1, keepdim=True)
    num = (pred_c * target_c).sum(dim=-1)
    den = torch.sqrt((pred_c**2).sum(dim=-1) * (target_c**2).sum(dim=-1) + 1e-8)
    return -(num / den).mean()


class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.y[i]


def mitsui_metric(y_true, y_pred):
    daily_corrs = []
    for i in range(len(y_true)):
        mask = ~np.isnan(y_true[i])
        if mask.sum() < 2:
            continue
        corr, _ = spearmanr(y_true[i, mask], y_pred[i, mask])
        if not np.isnan(corr):
            daily_corrs.append(corr)
    daily_corrs = np.array(daily_corrs)
    return daily_corrs.mean() / (daily_corrs.std() + 1e-8)


In [4]:
# Data prep with proper holdout split

feature_cols = [c for c in train.columns        if c != 'date_id']
target_cols  = [c for c in train_labels.columns if c != 'date_id']

# Merge features with labels and sort by date
df = train.merge(train_labels, on='date_id', how='inner')\
          .sort_values('date_id').reset_index(drop=True)

# Forward-fill only (no bfill, since that leaks future into past)
df[feature_cols] = df[feature_cols].ffill().fillna(0)
# For training, fill NaN targets with 0. We'll use raw train_labels for scoring.
df_train_targets = df[target_cols].fillna(0)

# Split by date_id
train_mask   = df['date_id'] <  HOLDOUT_START
holdout_mask = df['date_id'] >= HOLDOUT_START

# Fit scaler on training only
scaler = StandardScaler()
X_train   = scaler.fit_transform(df.loc[train_mask,   feature_cols].values).astype(np.float32)
X_holdout = scaler.transform(    df.loc[holdout_mask, feature_cols].values).astype(np.float32)

y_train = df_train_targets.loc[train_mask].values.astype(np.float32)

# For scoring: use raw train_labels (preserves NaNs for masking in metric)
holdout_dates = df.loc[holdout_mask, 'date_id'].values
y_holdout_true = (train_labels[train_labels['date_id'].isin(holdout_dates)]
                  .sort_values('date_id')[target_cols].values)

# Within-train val split for early-stopping signal during training
split_idx = int(0.85 * len(X_train))

print(f"X_train {X_train.shape}, y_train {y_train.shape}")
print(f"X_holdout {X_holdout.shape}, y_holdout_true {y_holdout_true.shape}")
print(f"train val split at row {split_idx}")


X_train (1827, 557), y_train (1827, 424)
X_holdout (134, 557), y_holdout_true (134, 424)
train val split at row 1552


## Predictor interface

`fit(X, y, split_idx)`: train on scaled data. Rows < split_idx are train, rest val.
`predict_one(x_history)`: takes (history_len, n_features), returns (n_targets,).

In [5]:
class Predictor:
    history_len = 1

    def fit(self, X, y, split_idx):
        raise NotImplementedError

    def predict_one(self, x_history):
        raise NotImplementedError


In [6]:
class LSTMPredictor(Predictor):
    history_len = 30

    def __init__(self, n_features, n_targets,
                 hidden=64, layers=2, dropout=0.2,
                 epochs=10, lr=1e-3, weight_decay=1e-4, batch_size=64):
        self.n_features = n_features
        self.n_targets  = n_targets
        self.epochs     = epochs
        self.batch_size = batch_size
        self.device     = device

        class _Net(nn.Module):
            def __init__(s):
                super().__init__()
                s.lstm = nn.LSTM(n_features, hidden, num_layers=layers,
                                 dropout=dropout, batch_first=True)
                s.head = nn.Linear(hidden, n_targets)
            def forward(s, x):
                out, _ = s.lstm(x)
                return s.head(out[:, -1, :])

        self.model = _Net().to(self.device)
        self.optimizer = torch.optim.Adam(self.model.parameters(),
                                          lr=lr, weight_decay=weight_decay)

    def _windows(self, X, y):
        Xs, ys = [], []
        for i in range(len(X) - self.history_len):
            Xs.append(X[i:i + self.history_len])
            ys.append(y[i + self.history_len])
        return np.array(Xs), np.array(ys)

    def fit(self, X, y, split_idx):
        X_seq, y_seq = self._windows(X, y)
        split_w = max(0, split_idx - self.history_len)
        X_tr, X_val = X_seq[:split_w], X_seq[split_w:]
        y_tr, y_val = y_seq[:split_w], y_seq[split_w:]

        train_loader = DataLoader(SeqDataset(X_tr, y_tr),
                                  batch_size=self.batch_size, shuffle=True)
        val_loader   = DataLoader(SeqDataset(X_val, y_val),
                                  batch_size=self.batch_size, shuffle=False)

        for epoch in range(self.epochs):
            self.model.train()
            tr_loss = 0.0
            for xb, yb in train_loader:
                xb, yb = xb.to(self.device), yb.to(self.device)
                self.optimizer.zero_grad()
                loss = pearson_loss(self.model(xb), yb)
                loss.backward()
                self.optimizer.step()
                tr_loss += loss.item()

            self.model.eval()
            v_loss = 0.0
            with torch.no_grad():
                for xb, yb in val_loader:
                    xb, yb = xb.to(self.device), yb.to(self.device)
                    v_loss += pearson_loss(self.model(xb), yb).item()

            print(f"Epoch {epoch+1:>2} | "
                  f"train {tr_loss/len(train_loader):.4f} | "
                  f"val {v_loss/len(val_loader):.4f}")

    def predict_one(self, x_history):
        self.model.eval()
        with torch.no_grad():
            x = torch.from_numpy(x_history).unsqueeze(0).to(self.device)
            return self.model(x).cpu().numpy()[0]


In [7]:
import lightgbm as lgb

class LightGBMPredictor(Predictor):
    history_len = 1

    def __init__(self, n_features, n_targets,
                 n_estimators=100, learning_rate=0.05, max_depth=6, **kwargs):
        self.n_features = n_features
        self.n_targets  = n_targets
        self.params = dict(n_estimators=n_estimators,
                           learning_rate=learning_rate,
                           max_depth=max_depth,
                           verbose=-1, **kwargs)
        self.models = []

    def fit(self, X, y, split_idx):
        X_tr, y_tr   = X[:split_idx], y[:split_idx]
        X_val, y_val = X[split_idx:], y[split_idx:]
        self.models = []
        for t in range(self.n_targets):
            m = lgb.LGBMRegressor(**self.params)
            m.fit(X_tr, y_tr[:, t],
                  eval_set=[(X_val, y_val[:, t])],
                  callbacks=[lgb.early_stopping(10, verbose=False)])
            self.models.append(m)
            if (t + 1) % 50 == 0:
                print(f"trained {t+1}/{self.n_targets}")

    def predict_one(self, x_history):
        x = x_history[-1:]
        return np.array([m.predict(x)[0] for m in self.models])


## Pick a model and train

In [8]:
predictor = LSTMPredictor(
    n_features=len(feature_cols),
    n_targets=len(target_cols),
    hidden=64, layers=2, dropout=0.2,
    epochs=10, lr=1e-3, weight_decay=1e-4,
)

# predictor = LightGBMPredictor(
#     n_features=len(feature_cols),
#     n_targets=len(target_cols),
#     n_estimators=100,
# )

predictor.fit(X_train, y_train, split_idx)


Epoch  1 | train -0.0159 | val -0.0229
Epoch  2 | train -0.0585 | val -0.0356
Epoch  3 | train -0.0901 | val -0.0462
Epoch  4 | train -0.1215 | val -0.0421
Epoch  5 | train -0.1547 | val -0.0416
Epoch  6 | train -0.1804 | val -0.0170
Epoch  7 | train -0.2054 | val -0.0274
Epoch  8 | train -0.2334 | val -0.0021
Epoch  9 | train -0.2512 | val -0.0136
Epoch 10 | train -0.2647 | val 0.0122


## Predict the holdout day-by-day

Same rolling-buffer logic the Kaggle gateway used. Just done in a normal loop.

In [9]:
HISTORY_LEN = predictor.history_len

# Seed buffer with the last HISTORY_LEN training rows
buffer = deque(X_train[-HISTORY_LEN:].astype(np.float32), maxlen=HISTORY_LEN)

predictions = []
for i in range(len(X_holdout)):
    buffer.append(X_holdout[i])
    x_history = np.stack(list(buffer), axis=0)
    pred = predictor.predict_one(x_history)
    predictions.append(pred)

predictions = np.array(predictions)
print(f"predictions {predictions.shape}, holdout truth {y_holdout_true.shape}")


predictions (134, 424), holdout truth (134, 424)


## Score

In [10]:
score = mitsui_metric(y_holdout_true, predictions)
print(f"Sharpe-variant score on holdout: {score:.4f}")


Sharpe-variant score on holdout: -0.1679
